# 05c — Uncapacitated P-Median Benchmark (Greedy Construction)

**Third of three companion notebooks.** No eⱼ, no K, no capacity constraint -- p sites
are opened, every LSOA assigned to its nearest open site.

**Method note:** the exact MILP version of this model (binary/integer zj + yij, solved
with PuLP/CBC) repeatedly timed out ("Not Solved") at Greater London scale even after
relaxing yij to continuous and allowing a 2% optimality gap -- the assignment matrix
(~4,994 x 4,994) is simply too large for branch-and-bound to finish in reasonable time,
regardless of how the capacity side of the problem is formulated. This notebook replaces
the MILP with a **greedy construction heuristic**, a standard, citable method in location
science (Daskin, 1995; originally due to Kuehn and Hamburger, 1963, for exactly this
problem): sites are added one at a time, each time picking whichever remaining candidate
most reduces total demand-weighted distance. It is not guaranteed to find the exact
optimum, but is well known to get close in practice, and here runs in minutes rather than
timing out.

**Efficiency note:** because greedy construction adds sites incrementally, the p=250
solution is literally the first 250 sites chosen on the way to p=500. Each alpha scenario
is therefore only constructed once (500 steps), with results recorded at checkpoints
p=100, 250, 500 -- not rebuilt three separate times.

## 0. Setup and reload cleaned data

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
import time
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"

demand_london = pd.read_csv(os.path.join(BASE, "05_processed/demand_london.csv"))
imd_london    = pd.read_csv(os.path.join(BASE, "05_processed/imd_london_clean.csv"))
census_london = pd.read_csv(os.path.join(BASE, "05_processed/census_london_clean.csv"))

outputs_dir = os.path.join(BASE, "06_outputs")
os.makedirs(outputs_dir, exist_ok=True)

print("Datasets reloaded.")
print(f"demand_london: {demand_london.shape}, imd_london: {imd_london.shape}")

Datasets reloaded.
demand_london: (4994, 16), imd_london: (4994, 10)


## 1. LSOA centroids (I = J) + fixed weights Vi and income_decile for M1-M4

In [2]:
lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None or lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand_london["lsoa_code"])
lsoa_london = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_london = lsoa_london.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]

lsoa_london["centroid"] = lsoa_london.geometry.centroid
lsoa_london["cx"] = lsoa_london["centroid"].x
lsoa_london["cy"] = lsoa_london["centroid"].y

lsoa_master = lsoa_london[["lsoa_code", "cx", "cy"]].merge(
    demand_london[["lsoa_code", "D_A", "D_B", "D_C", "D_D"]], on="lsoa_code", how="inner"
).merge(
    census_london[["lsoa_code", "Hi", "Ci"]], on="lsoa_code", how="left"
).merge(
    imd_london[["lsoa_code", "income_score", "income_decile"]], on="lsoa_code", how="left"
).reset_index(drop=True)
lsoa_master["Vi"] = lsoa_master["Hi"] * lsoa_master["Ci"]

n = len(lsoa_master)
coords = lsoa_master[["cx", "cy"]].to_numpy()
print(f"LSOA master table: {n} LSOAs (should be ~4,994)")
lsoa_master.head()

LSOA master table: 4994 LSOAs (should be ~4,994)


,lsoa_code,cx,cy,D_A,D_B,D_C,D_D,Hi,Ci,income_score,income_decile,Vi
0,E01000001,532151.194178,181615.201393,41.589935,40.636031,38.860358,37.241427,837.0,0.395460,0.013,10,331.0
1,E01000002,532443.686041,181645.724884,39.510571,38.623636,36.972625,35.467352,824.0,0.359223,0.018,10,296.0
2,E01000003,532207.014825,182030.129598,27.642857,27.262396,26.554175,25.908469,1017.0,0.216323,0.107,8,220.0
3,E01000005,533618.348516,181157.354255,14.952273,14.898217,14.797595,14.705854,479.0,0.248434,0.211,5,119.0
4,E01000006,544934.369716,184297.546524,64.835065,65.435782,66.554004,67.573518,554.0,0.931408,0.343,3,516.0


## 2. Full distance matrix

At n~4,994 this is ~25M float32 entries (~100MB) -- manageable in memory, and lets the
greedy heuristic below be fully vectorised (no per-pair Python loops).

In [3]:
t0 = time.time()
dist_matrix = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)).astype(np.float32)
print(f"Distance matrix built: {time.time()-t0:.1f}s, shape={dist_matrix.shape}, "
      f"{dist_matrix.nbytes/1e6:.0f}MB")

Distance matrix built: 0.6s, shape=(4994, 4994), 100MB


## 3. Greedy construction heuristic

Standard "myopic"/greedy-add algorithm for p-median (Kuehn & Hamburger, 1963; see also
Daskin, 1995): starting from no open sites, repeatedly add whichever remaining candidate
reduces total demand-weighted distance the most, until p sites are open. Vectorised over
all candidate sites at each step -- no MILP solver, no risk of Infeasible/timeout.

In [4]:
def greedy_p_median(demand_col, p_checkpoints, lsoa_master, dist_matrix):
    """
    Builds up to max(p_checkpoints) sites one at a time. Returns, for each checkpoint p,
    the set of open sites, each LSOA's assigned site and distance, and M1-M4.
    """
    Di = lsoa_master[demand_col].to_numpy().astype(np.float32)
    n_ = len(lsoa_master)
    p_checkpoints = sorted(p_checkpoints)
    p_max = p_checkpoints[-1]

    best_dist = np.full(n_, np.inf, dtype=np.float32)
    best_site = np.full(n_, -1, dtype=np.int64)
    remaining = np.ones(n_, dtype=bool)
    selected = []
    checkpoint_results = {}

    for step in range(p_max):
        improvement = np.maximum(0, best_dist[:, None] - dist_matrix)
        reduction = (Di[:, None] * improvement).sum(axis=0)
        reduction[~remaining] = -np.inf
        best_j = int(np.argmax(reduction))

        improved_mask = dist_matrix[:, best_j] < best_dist
        best_dist[improved_mask] = dist_matrix[improved_mask, best_j]
        best_site[improved_mask] = best_j

        selected.append(best_j)
        remaining[best_j] = False

        if (step + 1) in p_checkpoints:
            zj = np.zeros(n_, dtype=int)
            zj[selected] = 1
            checkpoint_results[step + 1] = {
                "zj": zj.copy(),
                "assigned_j": best_site.copy(),
                "dist": best_dist.copy(),
                "objective": float((Di * best_dist).sum()),
            }
            print(f"  checkpoint p={step+1}: objective={checkpoint_results[step+1]['objective']:,.0f}")

    return checkpoint_results


def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n_ = len(x)
    if x.sum() == 0:
        return 0.0
    cum = np.cumsum(x)
    return (n_ + 1 - 2 * (cum.sum() / cum[-1])) / n_


def compute_metrics(dist, lsoa_master):
    """M1-M4 exactly as defined in proposal Section 5.5."""
    Vi = lsoa_master["Vi"].to_numpy()
    decile = lsoa_master["income_decile"].to_numpy()

    M1 = float((Vi * dist).sum() / Vi.sum())
    within_800 = (dist < 800).astype(float)
    M2 = float((Vi * within_800).sum() / Vi.sum())

    def coverage_for_decile(d):
        mask = decile == d
        if mask.sum() == 0 or Vi[mask].sum() == 0:
            return np.nan
        return float((Vi[mask] * within_800[mask]).sum() / Vi[mask].sum())

    cov_d1, cov_d10 = coverage_for_decile(1), coverage_for_decile(10)
    M3 = cov_d1 - cov_d10

    eps = 1.0
    accessibility = 1.0 / (dist + eps)
    M4 = gini(accessibility)

    return {"M1_avg_dist_m": M1, "M2_coverage_800m": M2, "M3_imd_gap": M3, "M4_gini": M4}

## 4. Run all 4 alpha scenarios (each constructed once, up to p=500, with checkpoints
at p=100/250/500)

In [5]:
ALPHA_LABELS = {"A": "D_A", "B": "D_B", "C": "D_C", "D": "D_D"}
P_CHECKPOINTS = [100, 250, 500]

core_results = {}
core_summary_rows = []

for alpha_label, demand_col in ALPHA_LABELS.items():
    print(f"=== Building greedy solution: Scenario {alpha_label} (up to p=500) ===")
    t0 = time.time()
    checkpoints = greedy_p_median(demand_col, P_CHECKPOINTS, lsoa_master, dist_matrix)
    print(f"  total time: {time.time()-t0:.1f}s")

    for p, res in checkpoints.items():
        metrics = compute_metrics(res["dist"], lsoa_master)
        core_results[(alpha_label, p)] = {**res, **metrics}
        core_summary_rows.append({"scenario": alpha_label, "p": p, "objective": res["objective"], **metrics})

core_summary = pd.DataFrame(core_summary_rows)
print()
print("=== Core grid: M1-M4 for all 12 configurations ===")
print(core_summary.to_string(index=False))

=== Building greedy solution: Scenario A (up to p=500) ===
  checkpoint p=100: objective=505,602,016
  checkpoint p=250: objective=314,279,104
  checkpoint p=500: objective=215,590,448
  total time: 13.4s
=== Building greedy solution: Scenario B (up to p=500) ===
  checkpoint p=100: objective=504,933,504
  checkpoint p=250: objective=314,696,576
  checkpoint p=500: objective=215,602,304
  total time: 13.1s
=== Building greedy solution: Scenario C (up to p=500) ===
  checkpoint p=100: objective=506,709,312
  checkpoint p=250: objective=314,808,640
  checkpoint p=500: objective=215,184,208
  total time: 12.9s
=== Building greedy solution: Scenario D (up to p=500) ===
  checkpoint p=100: objective=505,086,848
  checkpoint p=250: objective=313,349,376
  checkpoint p=500: objective=215,173,344
  total time: 13.0s

=== Core grid: M1-M4 for all 12 configurations ===
scenario   p   objective  M1_avg_dist_m  M2_coverage_800m  M3_imd_gap  M4_gini
       A 100 505602016.0    1403.657364          

## 5. Save results

In [6]:
results_wide = lsoa_master[["lsoa_code"]].copy()
for (alpha_label, p), result in core_results.items():
    results_wide[f"zj_{alpha_label}_p{p}"] = result["zj"]

output_path = os.path.join(BASE, "05_processed/uncapacitated_pmedian_results.csv")
results_wide.to_csv(output_path, index=False)
print(f"Saved per-LSOA zj (12 configurations): {output_path}")

core_summary.to_csv(os.path.join(BASE, "06_outputs/uncapacitated_core_grid_summary.csv"), index=False)
print("Saved: uncapacitated_core_grid_summary.csv")

Saved per-LSOA zj (12 configurations): /Users/alexia/Documents/CASA/Dissertation/05_processed/uncapacitated_pmedian_results.csv
Saved: uncapacitated_core_grid_summary.csv


## 6. Diagnostics — equity mechanism check

In [7]:
print("=== Pearson r: zj (site opened) vs income_score, by scenario (p=250) ===")
diag_table = results_wide.merge(lsoa_master[["lsoa_code", "income_score"]], on="lsoa_code", how="left")
for alpha_label in ["A", "B", "C", "D"]:
    col = f"zj_{alpha_label}_p250"
    r, p_val = pearsonr(diag_table[col], diag_table["income_score"])
    print(f"Scenario {alpha_label}: r = {r:.4f}, p = {p_val:.4g}")

=== Pearson r: zj (site opened) vs income_score, by scenario (p=250) ===
Scenario A: r = -0.0217, p = 0.1248
Scenario B: r = -0.0297, p = 0.03614
Scenario C: r = -0.0185, p = 0.1915
Scenario D: r = -0.0181, p = 0.201


## 7. Comparison note

This notebook's M1-M4 (no capacity, no eⱼ, no K, solved by greedy construction rather
than exact MILP) is the benchmark against which 05b_two_stage_model.ipynb (adds eⱼ + K
+ capacity) can be read as an ablation. 05_p_median_715.ipynb (joint MILP + slack) shows
why a literal joint capacitated version is not solvable in a meaningful way at this
scale -- and this notebook's own history (exact MILP timing out even without any
capacity terms at all) shows that the *assignment* problem alone is already at the edge
of what exact MILP can handle here, independent of capacity.